<a href="https://colab.research.google.com/github/kennedy336/Machine-Learning-and-Deep-learning-mini-Projects/blob/main/BERT_FINE_TUNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [ ]:
  # TP — Fine-tuner un modele sur les minutes de la Fed (FOMC)
# Tache : Stance Detection  ->  hawkish / dovish / neutral / irrelevant
# Dataset : gtfintechlab/federal_reserve_system  (CC BY-NC-SA 4.0)

# !pip install -q transformers datasets evaluate scikit-learn accelerate

import numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          TrainingArguments, Trainer,
                          DataCollatorWithPadding,
                          EarlyStoppingCallback)
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)

MODELE = "distilbert-base-uncased"
SUBSET = "5768"          # 3 sous-ensembles : 5768 / 78516 / 944601
CIBLE = "stance_label"   # ou "time_label" / "certain_label"


In [ ]:
# 1. CHARGER LES DONNEES
ds = load_dataset("gtfintechlab/federal_reserve_system", SUBSET)
print(ds)
print(ds["train"][0]["sentences"])
print(ds["train"][0][CIBLE])

# les labels sont du TEXTE -> on les convertit en entiers
labels = sorted(set(ds["train"][CIBLE]))
l2id = {l: i for i, l in enumerate(labels)}
id2l = {i: l for l, i in l2id.items()}
print("Labels :", l2id)

# 2. TOKENISER
tok = AutoTokenizer.from_pretrained(MODELE)

def prep(batch):
    enc = tok(batch["sentences"], truncation=True, max_length=128)
    enc["labels"] = [l2id[l] for l in batch[CIBLE]]
    return enc

ds_tok = ds.map(prep, batched=True, remove_columns=ds["train"].column_names)
collator = DataCollatorWithPadding(tokenizer=tok)


README.md:   0%|          | 0.00/4.28k [00:00<?, ?B/s]

5768/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 78.1kB            

5768/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

5768/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.1kB            

5768/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
# 3. LE MODELE (corps pre-entraine + tete neuve)
model = AutoModelForSequenceClassification.from_pretrained(
    MODELE, num_labels=len(labels), id2label=id2l, label2id=l2id)

# Option A - tout geler (feature extraction) : decommentez ces 2 lignes
# for p in model.distilbert.parameters():
#     p.requires_grad = False

n_tot = sum(p.numel() for p in model.parameters())
n_ent = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Poids entraines : {n_ent:,} / {n_tot:,} ({100*n_ent/n_tot:.1f} %)")

# 4. LA METRIQUE (F1 macro : classes desequilibrees)
def metriques(p):
    pred = np.argmax(p.predictions, axis=-1)
    return {"accuracy": accuracy_score(p.label_ids, pred),
            "f1": f1_score(p.label_ids, pred, average="macro")}


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Poids entraines : 66,956,548 / 66,956,548 (100.0 %)


In [ ]:
# 5. ENTRAINEMENT
args = TrainingArguments(
    output_dir="sortie_fomc",
    learning_rate=2e-5,              # petit : on ajuste, pas de zero
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",           # anciennes versions : evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,     # = early stopping
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=20,
    seed=42)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["val"],
    processing_class=tok,            # anciennes versions : tokenizer=tok
    data_collator=collator,
    compute_metrics=metriques,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.219204,1.166195,0.400000,0.219485
2,1.092343,1.062542,0.520000,0.518782
3,0.971118,1.011529,0.493333,0.518059
4,0.824382,0.980581,0.540000,0.556397
5,0.614321,0.973366,0.520000,0.550444
6,0.482320,0.980117,0.560000,0.571620
7,0.397434,1.019132,0.553333,0.563066
8,0.320471,1.009712,0.540000,0.561203


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=352, training_loss=0.760807453231378, metrics={'train_runtime': 95.0792, 'train_samples_per_second': 58.898, 'train_steps_per_second': 3.702, 'total_flos': 96818493307200.0, 'train_loss': 0.760807453231378, 'epoch': 8.0})

In [ ]:
# 6. EVALUATION FINALE (le test : UNE seule fois)
print(trainer.evaluate(ds_tok["test"]))

sortie = trainer.predict(ds_tok["test"])
y_pred = np.argmax(sortie.predictions, axis=-1)
y_vrai = sortie.label_ids

print(classification_report(y_vrai, y_pred, target_names=labels, digits=3))
print(confusion_matrix(y_vrai, y_pred))

# baseline obligatoire : predire toujours la classe majoritaire
maj = np.bincount(ds_tok["train"]["labels"]).argmax()
base = np.full_like(y_vrai, maj)
print("Baseline F1 macro :",
      f1_score(y_vrai, base, average="macro", zero_division=0))

# 7. UTILISER LE MODELE
from transformers import pipeline

clf = pipeline("text-classification", model=model, tokenizer=tok)
print(clf("the committee decided to raise the target range "
          "by 25 basis points."))

trainer.save_model("mon_modele_fomc")
tok.save_pretrained("mon_modele_fomc")


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.320471,1.022129,8,0.560000,0.569944


{'eval_loss': 1.022128939628601, 'eval_accuracy': 0.56, 'eval_f1': 0.569944269505673}


              precision    recall  f1-score   support

      dovish      0.567     0.630     0.596        54
     hawkish      0.605     0.535     0.568        43
  irrelevant      0.667     0.571     0.615         7
     neutral      0.500     0.500     0.500        46

    accuracy                          0.560       150
   macro avg      0.585     0.559     0.570       150
weighted avg      0.562     0.560     0.560       150

[[34  7  0 13]
 [13 23  0  7]
 [ 0  0  4  3]
 [13  8  2 23]]
Baseline F1 macro : 0.1323529411764706
[{'label': 'neutral', 'score': 0.6741739511489868}]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('mon_modele_fomc/tokenizer_config.json', 'mon_modele_fomc/tokenizer.json')

In [ ]:
# 8. VARIANTE LoRA  (pip install peft)
# from peft import LoraConfig, get_peft_model
#
# base = AutoModelForSequenceClassification.from_pretrained(
#     MODELE, num_labels=len(labels), id2label=id2l, label2id=l2id)
# cfg = LoraConfig(r=16, lora_alpha=32,
#                  target_modules=["q_lin", "v_lin"],   # DistilBERT
#                  lora_dropout=0.05, task_type="SEQ_CLS")
# model_lora = get_peft_model(base, cfg)
# model_lora.print_trainable_parameters()
# -> puis on relance exactement le meme Trainer avec model_lora
